# Otto Group Product Classification — 9-class log-loss, honest multiclass

Case: [Otto Group Product Classification Challenge](https://www.kaggle.com/competitions/otto-group-product-classification-challenge)
(61,878 products, 93 anonymized integer count features, **9 imbalanced
classes**, scored on multi-class log loss).

Why this case rounds out the showcase:

- it is the **only multiclass** notebook — selection runs on
  `metric="log_loss"`, the exact competition metric, exercising the library's
  multiclass projection and per-class metric averaging;
- log loss *rewards calibrated probabilities*, so `calibrate="auto"` is
  on-metric here, not a tangential add-on;
- Otto is the textbook case where **HPO + ensembling** beat a single untuned
  model — the winning solutions were deep multi-level stacks — so it is the
  fairest test of whether level-2 machinery earns its keep.

> Data note: downloaded from a community Kaggle **mirror**
> (`msandipan98/otto-group-product-classification`); the competition files are
> identical. Live submission stays off — the comparison is against published
> leaderboard numbers, not a fresh submission.

In [1]:
import json
import logging
import os
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

from honestml import AutoML, CVConfig, render_report, save_run_report

SEED = 42
DATA = Path("data/otto")
RESULTS = Path("results/otto")
RESULTS.mkdir(parents=True, exist_ok=True)
KAGGLE = os.environ.get("KAGGLE_BIN") or shutil.which("kaggle") or str(Path.home() / ".local" / "bin" / "kaggle")

logging.basicConfig(format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("honestml").setLevel(logging.INFO)

In [2]:
if not (DATA / "train.csv").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [KAGGLE, "datasets", "download", "-d", "msandipan98/otto-group-product-classification", "-p", str(DATA), "--unzip"],
        check=True,
    )

In [3]:
df = pd.read_csv(DATA / "train.csv").drop(columns="id")
y = df.pop("target")
X = df
print(f"rows: {len(X)}, features: {X.shape[1]}, classes: {y.nunique()}")
print(y.value_counts().sort_index().to_string())
X.head()

rows: 61877, features: 93, classes: 9
target
Class_1     1929
Class_2    16122
Class_3     8004
Class_4     2691
Class_5     2739
Class_6    14135
Class_7     2839
Class_8     8464
Class_9     4954


,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,feat_10,...,feat_84,feat_85,feat_86,feat_87,feat_88,feat_89,feat_90,feat_91,feat_92,feat_93
0,1,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,1,6,1,5,0,0,1,...,22,0,1,2,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0


## Level 1 — the untuned multiclass default

The full zoo (baseline / linear / catboost / lightgbm / xgboost) on a 5-fold
stratified CV, selecting on out-of-fold **log loss** (lower is better), with a
20% stratified outer holdout scored exactly once and on-metric probability
calibration (`calibrate="auto"`).

In [4]:
model = AutoML(
    task="multiclass",
    metric="log_loss",
    cv=CVConfig(outer_holdout=0.2, calibrate="auto"),
    random_state=SEED,
)
t0 = time.perf_counter()
model.fit(X, y)
print(f"fit took {(time.perf_counter() - t0) / 60:.1f} min")

INFO honestml.adapters.reader: auto-typing column=feat_5 dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=feat_6 dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=feat_12 dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=feat_21 dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=feat_33 dtype=Int64 role=categorical reason=low_cardinality_int


...

WARNING honestml.adapters.boosting: boosting 'lightgbm' trained without early stopping; leaderboard comparison may favor overfit settings


INFO honestml: stage key=run stage=refit elapsed=9.9s


INFO honestml: stage key=run stage=refit elapsed=10.5s


INFO honestml: stage key=run stage=finalize elapsed=10.5s


fit took 25.5 min


In [5]:
report = model.run_report_
pd.DataFrame(report["leaderboard"])

,model_id,score,rank
0,lightgbm,0.496351,1
1,xgboost,0.501565,2
2,catboost,0.517749,3
3,linear,0.646301,4
4,baseline,1.950380,5


## The honesty check

Multiclass log loss is a proper scoring rule, so the only number worth trusting
is the one computed on data that took no part in selection: the untouched 20%
holdout, next to the out-of-fold selection score and their gap (the optimism).

In [6]:
selection = next(e["score"] for e in report["leaderboard"] if e["model_id"] == report["winner"])
print(f"winner          : {report['winner']}")
print(f"equivalence band: {report['band']['member_ids']}")
print(f"selection score : {selection:.4f}   (out-of-fold log loss, lower is better)")
print(f"holdout score   : {report['holdout_score']:.4f}   (untouched 20%, scored once)")
print(f"optimism        : {selection - report['holdout_score']:+.4f}")

winner          : lightgbm
equivalence band: ['lightgbm']
selection score : 0.4964   (out-of-fold log loss, lower is better)
holdout score   : 0.4672   (untouched 20%, scored once)
optimism        : +0.0291


In [7]:
save_run_report(report, RESULTS)
md_path = render_report(report, RESULTS, fmt="md")
Markdown(md_path.read_text(encoding="utf-8"))

# AutoML run report

## Run

| | |
|---|---|
| task | multiclass |
| metric | log_loss |
| winner | lightgbm |
| holdout_score | 0.467208 |
| honestml_version | 1.0.0 |
| run_fingerprint | f2d3323ca27c004807f43abd5ed2b8a200ab0648ff8fb83696f5be3ca054b13d |
| significance | bootstrap |
| preset | n/a |

## Equivalence band

| | |
|---|---|
| members | lightgbm |
| width | 1 |
| unstable | False |
| winner_by_tiebreak | False |

## Budget

| | |
|---|---|
| mode | none |
| exhausted | False |
| exhausted_by | n/a |
| skipped | n/a |

## Serving

| | |
|---|---|
| finalize | True |
| shipped_on | all |
| outer_holdout | 0.2 |

## Leaderboard

| rank | model | score |
|---|---|---|
| 1 | lightgbm (winner) | 0.496351 |
| 2 | xgboost | 0.501565 |
| 3 | catboost | 0.517749 |
| 4 | linear | 0.646301 |
| 5 | baseline | 1.95038 |

## Timings (s)

| stage | elapsed |
|---|---|
| run.selection | 1507.6 |
| run.refit | 10.5 |
| run.finalize | 10.5 |

## Resolved config

```json
{
  "seed": 42,
  "cv": {
    "scheme": "stratified",
    "n_splits": 5,
    "n_test": 1,
    "n_es": 1,
    "purge": 0,
    "embargo": 0,
    "period": null,
    "period_size": null,
    "step_periods": null,
    "purge_delta": null,
    "embargo_delta": null,
    "max_train_periods": null,
    "max_train_size": null,
    "weighting": "pooled",
    "calibrate": "auto",
    "selection": "raw",
    "refinement_min_oof": 2000,
    "outer_holdout": 0.2
  },
  "budget": {
    "mode": "none",
    "time_budget_s": null,
    "n_trials": null,
    "memory_limit_mb": null
  },
  "fe": {
    "target_encoding": false,
    "te_smoothing": 10.0,
    "frequency_encoding": false,
    "intersections": false,
    "max_pairs": 50
  },
  "fs": null,
  "hpo": null,
  "ensemble": null,
  "significance": "bootstrap",
  "run_mode": "full",
  "model_types": [
    "catboost",
    "lightgbm"
  ]
}
```


## Comparison with published results

Otto's winning solutions were famous for the size of their stacks. The
first-place team (Gilberto Titericz & Stanislav Semenov) reached roughly
**0.382 private log loss** with a multi-level ensemble of dozens of models;
their strongest individual second-level models (an NN and an XGBoost) each sat
around **0.39**, and a single well-tuned gradient boosting typically lands in
the **0.44–0.50** range. Read that ladder against the honest numbers above: the
gap between a single model and the leaderboard top is exactly the gap this
notebook's level-2 stage (HPO + significance-gated ensembling) is allowed to
try to close — honestly, and only if it survives the gate.

## Level 2 — HPO + ensembling + calibration (the levers that won Otto)

What changes against level 1:

- `preset="best"` turns on **significance-gated Caruana ensembling** — the
  single lever the Otto leaderboard rewarded most;
- an explicit `HPOConfig(n_trials=20, timeout_s=900)` — Optuna tuning capped at
  20 trials / 15 minutes per boosting type;
- `FeatureSelectionConfig(strategy="importance", cutoff="auto")` — how many of
  the 93 anonymous count features carry signal, with the honest no-selection
  gate that ships all features unless a subset is demonstrably better;
- feature engineering stays off (no categoricals — the features are integer
  counts); calibration and the untouched 20% holdout as in level 1.

In [8]:
from honestml import FeatureSelectionConfig, HPOConfig

model_full = AutoML(
    task="multiclass",
    metric="log_loss",
    preset="best",
    hpo=HPOConfig(n_trials=20, timeout_s=900),
    feature_selection=FeatureSelectionConfig(strategy="importance", cutoff="auto"),
    cv=CVConfig(outer_holdout=0.2, calibrate="auto"),
    random_state=SEED,
)
t0 = time.perf_counter()
model_full.fit(X, y)
print(f"level-2 fit took {(time.perf_counter() - t0) / 60:.1f} min")

INFO honestml.composition.presets: preset best applied: ['ensemble']


INFO honestml.adapters.reader: auto-typing column=feat_5 dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=feat_6 dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=feat_12 dtype=Int64 role=categorical reason=low_cardinality_int


INFO honestml.adapters.reader: auto-typing column=feat_21 dtype=Int64 role=categorical reason=low_cardinality_int


...

INFO honestml: stage key=run stage=ensemble elapsed=274.2s


INFO honestml: stage key=run stage=refit elapsed=15.1s


INFO honestml: stage key=run stage=refit elapsed=15.7s


INFO honestml: stage key=run stage=finalize elapsed=15.7s


level-2 fit took 121.9 min


In [9]:
report_full = model_full.run_report_
sel_full = next(e["score"] for e in report_full["leaderboard"] if e["model_id"] == report_full["winner"])
print(f"winner          : {report_full['winner']}")
print(f"selection score : {sel_full:.4f}   (level 1: {selection:.4f})")
print(f"holdout score   : {report_full['holdout_score']:.4f}   (level 1: {report['holdout_score']:.4f})")
print(f"optimism        : {sel_full - report_full['holdout_score']:+.4f}")
print()
print("preset  :", report_full["preset"])
print("hpo     :", json.dumps(report_full["hpo"], default=str)[:600])
print("fs      :", json.dumps(report_full["feature_selection"], default=str)[:400])
print("ensemble:", json.dumps(report_full["ensemble"], default=str)[:400])
pd.DataFrame(report_full["leaderboard"])

winner          : xgboost
selection score : 0.4727   (level 1: 0.4964)
holdout score   : 0.4522   (level 1: 0.4672)
optimism        : +0.0205

preset  : {'name': 'best', 'applied': ['ensemble']}
hpo     : {"backend": "optuna", "inner_cv": 3, "deterministic": false, "selection_oof_is_post_tuning": true, "tuned_on_full_feature_space": true, "cost_estimate_fits": 180, "tuned": {"catboost": {"chosen_params": {"depth": 10, "learning_rate": 0.06690992453172909, "iterations": 500, "l2_leaf_reg": 1.042225933947968, "subsample": 0.8607466203112714, "one_hot_max_size": 4}, "inner_best_score": 0.5089372516460592, "n_trials_run": 11}, "lightgbm": {"chosen_params": {"max_depth": 10, "learning_rate": 0.09470745919646917, "n_estimators": 350, "reg_lambda": 5.53165764374762, "subsample": 0.6684815593910434, "c
fs      : {"strategy": "importance", "n_selected": 93, "selected": ["feat_1", "feat_2", "feat_3", "feat_4", "feat_7", "feat_8", "feat_9", "feat_10", "feat_11", "feat_13", "feat_14", "feat_15", "

,model_id,score,rank
0,xgboost,0.472719,1
1,lightgbm,0.483962,2
2,catboost,0.496673,3
3,linear,0.646301,4
4,baseline,1.950380,5


## Level 1 vs level 2: what the full pipeline buys

The full zoo runs cleanly on the 9-class target — **all five models**, xgboost
included. (xgboost 3.x requires contiguous `0..K-1` labels, so the library codes
the `Class_*` strings internally and decodes on the way out — ADR-0081.) Level 1
ranks **lightgbm** first (OOF log loss **0.4964**), with **xgboost** (0.5016) and
catboost (0.5177) close behind, the median-imputed linear model (0.6463) further
back and the stratified baseline (1.95) far away. The optimism is mild
(selection 0.4964 vs holdout 0.4672) — the holdout comes out slightly better
than the OOF estimate, which is honest, not a leak.

**This is the notebook where level 2 earns its keep.** After 20 Optuna trials,
**xgboost takes the lead** — OOF log loss **0.4727** (level 1: 0.4964) and holdout
**0.4522** (level 1: 0.4672). Unlike the first four showcases, where the honest
contour shows the extra machinery buying nothing, here HPO genuinely moves the
number: a tuned xgboost beats every untuned default, and the gain survives onto
the untouched holdout (optimism +0.0205). The model that trails the untuned
lightgbm at level 1 is the one that wins once it is tuned.

The other two level-2 levers stay **off, honestly**:

- **feature selection** ships all 93 features — its `importance` / `cutoff=auto`
  subset does not clear the no-selection gate, because the anonymized count
  features are not redundant enough;
- **Caruana ensembling** is gated off (`gate_reason: worse_than_best`,
  `oof_delta` 0.022) — the blend, led by lightgbm with linear and xgboost close
  behind, is not significantly better than the single tuned xgboost, so the
  simpler model ships.

Against the published ladder — Otto's winning multi-level stacks reached ~0.39,
a single well-tuned gradient boosting lands ~0.44–0.50 — the shipped holdout
**0.4522** sits exactly where one honest, calibrated, tuned single model should:
no leaderboard-chasing, and every claim backed by a number that took no part in
the choice.